# Explainable AI for Heart Disease Risk Prediction
### A human-centred XAI pipeline — data → model → explanation

**BSc precursor project for:** *Human-Centred Design Frameworks for Explainable AI in Healthcare Applications* (proposed MSc research)

This notebook implements the technical core of the project:

1. Load and clean the UCI Heart Disease dataset
2. Exploratory data analysis (EDA)
3. Train and compare classification models
4. Evaluate predictive performance
5. Generate SHAP explanations (global + local)
6. Save the trained model and explainer artifacts for the Streamlit prototype (`app/app.py`)

> **Run this notebook top-to-bottom.** Every plot and number you see after running it is real output from your own run — write those numbers into your report, not any example numbers you may have seen elsewhere.


## 0. Setup

**Before running this notebook**, get the dataset (choose ONE option):

**Option A — `ucimlrepo` (recommended, pulls directly from UCI):**
```bash
pip install ucimlrepo
```

**Option B — manual download:**
Download the Heart Disease dataset from the UCI Machine Learning Repository
(https://archive.ics.uci.edu/dataset/45/heart+disease) or one of its Kaggle mirrors
(search "Heart Disease UCI" on Kaggle), and save it as `../data/heart.csv` relative
to this notebook, with columns:
`age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target`

Install the remaining requirements from `requirements.txt`:
```bash
pip install -r ../requirements.txt
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/heart.csv")
ARTIFACT_DIR = Path("../app/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load the data

In [ ]:
def load_heart_disease():
    """Load the UCI Heart Disease dataset, preferring ucimlrepo, falling back to a local CSV."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=45)
        X = ds.data.features.copy()
        y = ds.data.targets.copy()
        df = pd.concat([X, y], axis=1)
        # the ucimlrepo target column may be named 'num' — standardise to 'target'
        target_col = [c for c in df.columns if c.lower() in ("num", "target", "goal")][0]
        df = df.rename(columns={target_col: "target"})
        print(f"Loaded {len(df)} rows via ucimlrepo.")
        return df
    except Exception as e:
        print(f"ucimlrepo unavailable or failed ({e}). Trying local CSV at {DATA_PATH} ...")
        if DATA_PATH.exists():
            df = pd.read_csv(DATA_PATH)
            print(f"Loaded {len(df)} rows from {DATA_PATH}.")
            return df
        raise FileNotFoundError(
            "Could not load the dataset automatically.\n"
            "1) pip install ucimlrepo, OR\n"
            f"2) download the CSV manually and place it at {DATA_PATH.resolve()}"
        )

df_raw = load_heart_disease()
df_raw.head()


## 2. Initial inspection

Check shape, dtypes, and missing values before doing anything else. The original UCI files
encode missing values as `"?"` in the `ca` and `thal` columns — these will show up as
non-numeric entries or `NaN` depending on how you loaded the data.

In [ ]:
print("Shape:", df_raw.shape)
df_raw.info()


In [ ]:
df_raw.isna().sum()


## 3. Data cleaning

Steps:
1. Coerce all feature columns to numeric (this turns any stray `"?"` into `NaN`).
2. Impute the small number of missing values in `ca` and `thal` with the column median
   (documented, reproducible choice — justify this in your report; with so few missing
   rows, dropping them instead is an equally defensible alternative).
3. Binarise the target: the raw target ranges 0 (no disease) to 4 (severity of disease).
   Following standard practice for this dataset, collapse 1–4 into a single "disease present" class.


In [ ]:
df = df_raw.copy()

feature_cols = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                 "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

missing_before = df[feature_cols].isna().sum().sum()
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))
print(f"Imputed {missing_before} missing values using column medians.")

df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)
df["target"] = (df["target"] > 0).astype(int)  # 0 = no disease, 1 = disease present

print(df["target"].value_counts(normalize=True).rename("proportion"))
df.head()


## 4. Exploratory data analysis

Look at class balance, feature distributions, and correlations before modelling.
Record 2–3 observations from this section in your report (Chapter: Results / EDA) —
this is genuine analysis of YOUR data, not something to copy from elsewhere.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
df["target"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No disease (0)", "Disease present (1)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class balance")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[feature_cols + ["target"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["age", "thalach", "oldpeak"]):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_xticklabels(["No disease", "Disease"])
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 5. Feature engineering & train/test split

The raw columns are mostly already numeric/ordinal-encoded by UCI, so minimal engineering
is required — the main technical value of this project is in the modelling → explanation →
evaluation pipeline, not heavy feature construction. We do:

- An 80/20 stratified train/test split (stratified on `target` to preserve class balance).
- Standardisation (`StandardScaler`) fitted on the training set only, for the linear/SVM models.
  Tree-based models use the unscaled features (they are scale-invariant).

In [ ]:
X = df[feature_cols]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 6. Model training & comparison

We compare four models spanning the interpretability–performance spectrum:

| Model | Role in the comparison |
|---|---|
| Logistic Regression | Fully transparent baseline — coefficients are directly interpretable |
| Random Forest | Non-linear, robust, has built-in feature importances, supports fast exact SHAP (`TreeExplainer`) |
| Gradient Boosting | Usually the strongest tabular performer here; also supports `TreeExplainer` |
| SVM (RBF kernel) | Non-linear "true black box" reference — good contrast case for why post-hoc XAI (SHAP) matters |

Comparing these lets you discuss, with real evidence, whether higher predictive performance
comes at the cost of interpretability for this task (a core theme for your MSc direction).

In [ ]:
models = {
    "LogisticRegression": (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), True),
    "RandomForest": (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE), False),
    "GradientBoosting": (GradientBoostingClassifier(random_state=RANDOM_STATE), False),
    "SVM_RBF": (SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE), True),
}

results = {}
fitted_models = {}

for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train
    Xte = X_test_scaled if needs_scaling else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]

    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    }
    fitted_models[name] = model

results_df = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)
results_df.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, (model, needs_scaling) in models.items():
    Xte = X_test_scaled if needs_scaling else X_test
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.2f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves — model comparison")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 7. Model selection

Pick the best model **on your own results** (don't hardcode an assumption). We select
by ROC-AUC on the held-out test set, but note in your report if a different model would be
preferable clinically (e.g. prioritising recall, since missing a positive case is usually
costlier than a false alarm).

In [ ]:
best_model_name = results_df.index[0]
best_model = fitted_models[best_model_name]
best_needs_scaling = models[best_model_name][1]
print(f"Best model by ROC-AUC: {best_model_name}")
results_df.loc[[best_model_name]].round(3)


In [ ]:
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled if best_needs_scaling else X_test))
disp = ConfusionMatrixDisplay(cm, display_labels=["No disease", "Disease"])
disp.plot(cmap="Blues")
plt.title(f"Confusion matrix — {best_model_name}")
plt.show()


## 8. Explainability with SHAP

We use **SHAP (SHapley Additive exPlanations)** because:
- It has a solid game-theoretic foundation (Shapley values), giving each feature a
  fair, additive contribution to a prediction.
- `TreeExplainer` is exact and fast for tree-based models (Random Forest / Gradient Boosting).
- It produces both **global** explanations (which features matter overall — useful for model
  validation and clinician-level trust) and **local** explanations (why THIS patient got THIS
  prediction — the form of explanation your human evaluation study will actually test).

If the best model above is `SVM_RBF` or another non-tree model, `shap.KernelExplainer` is used
instead (slower, model-agnostic, approximate).

In [ ]:
def get_positive_class_shap(shap_values):
    """Normalise SHAP output across SHAP versions/explainers to a (n_samples, n_features) array
    for the positive class."""
    arr = np.array(shap_values)
    if isinstance(shap_values, list):
        return shap_values[1]
    if arr.ndim == 3:
        return arr[:, :, 1]
    return arr


explain_X_test = X_test_scaled if best_needs_scaling else X_test
explain_X_train = X_train_scaled if best_needs_scaling else X_train

if best_model_name in ("RandomForest", "GradientBoosting"):
    explainer = shap.TreeExplainer(best_model)
    raw_shap_values = explainer.shap_values(explain_X_test)
else:
    background = shap.sample(explain_X_train, 100, random_state=RANDOM_STATE)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    raw_shap_values = explainer.shap_values(explain_X_test, nsamples=200)

shap_values_pos = get_positive_class_shap(raw_shap_values)
print("SHAP values shape:", shap_values_pos.shape)


### 8a. Global explanation — which features matter overall?

The beeswarm plot below ranks features by mean absolute SHAP value (top = most influential
overall). Each dot is one test patient; colour shows whether that patient's value for that
feature was high (red) or low (blue), and position shows whether it pushed the prediction
towards "disease" (right) or "no disease" (left).

In [ ]:
shap.summary_plot(shap_values_pos, explain_X_test, show=False)
plt.title(f"Global feature importance (SHAP) — {best_model_name}")
plt.tight_layout()
plt.show()


### 8b. Local explanation — why did the model predict this for ONE patient?

Pick a single test-set patient and show exactly how each feature pushed their predicted
risk up or down from the average. This is the explanation format your Streamlit prototype
will show to end users, and the object of your human-centred evaluation.

In [ ]:
example_idx = 0  # change this to inspect different patients
example_row = explain_X_test.iloc[[example_idx]]

if best_model_name in ("RandomForest", "GradientBoosting"):
    base_value = explainer.expected_value
    base_value = base_value[1] if isinstance(base_value, (list, np.ndarray)) and np.ndim(base_value) > 0 else base_value
else:
    base_value = explainer.expected_value[1] if np.ndim(explainer.expected_value) > 0 else explainer.expected_value

explanation = shap.Explanation(
    values=shap_values_pos[example_idx],
    base_values=base_value,
    data=example_row.iloc[0].values,
    feature_names=feature_cols,
)

shap.plots.waterfall(explanation, show=False)
plt.tight_layout()
plt.show()

predicted_proba = best_model.predict_proba(example_row)[0, 1]
actual_label = y_test.iloc[example_idx]
print(f"Predicted probability of disease: {predicted_proba:.2f}")
print(f"Actual label: {'Disease' if actual_label == 1 else 'No disease'}")


**Write your own interpretation here** (this is exactly the kind of thing a supervisor
will ask you to explain in an interview):

- Which 2–3 features pushed this prediction up? Which pushed it down?
- Do the top global features (8a) match established cardiac risk factors from the clinical
  literature (e.g. chest pain type, max heart rate, ST depression, number of major vessels)?
  This is your **sanity check** that the model is learning something clinically plausible,
  not spurious correlations — discuss this explicitly in your report's Discussion section.


## 9. Save artifacts for the Streamlit prototype

Save the trained model, scaler, feature list, a small background sample (needed for SHAP
inside the app), and the metadata the app needs — so `app/app.py` doesn't need to retrain
anything.

In [ ]:
artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "needs_scaling": best_needs_scaling,
    "scaler": scaler,
    "feature_cols": feature_cols,
    "background_sample": (X_train_scaled if best_needs_scaling else X_train).sample(
        n=min(100, len(X_train)), random_state=RANDOM_STATE
    ),
    "test_metrics": results_df.loc[best_model_name].to_dict(),
}

with open(ARTIFACT_DIR / "model_bundle.pkl", "wb") as f:
    pickle.dump(artifact, f)

results_df.round(4).to_csv(ARTIFACT_DIR / "model_comparison_results.csv")

print("Saved:")
print(" -", ARTIFACT_DIR / "model_bundle.pkl")
print(" -", ARTIFACT_DIR / "model_comparison_results.csv")


## 10. Summary & next steps

- [ ] Fill in your own observations from Sections 4, 6 and 8 into your technical report
      (see `docs/report_template.md`).
- [ ] Screenshot the plots in this notebook for your evidence portfolio.
- [ ] Run `streamlit run app/app.py` to launch the interactive prototype built on the
      artifacts saved above.
- [ ] Use the prototype to run the human-centred evaluation described in
      `docs/evaluation_questionnaire.md`.

### Dataset citation
Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1988). Heart Disease
[Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X
